# Fenestration Graph
A dual graph extended to include doors and windows as explicit nodes.  
Every room, door, and window is a node. Edges connect each aperture to the rooms it serves.

## 1. Import Libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color

c:\Users\danie\Documents\GitHub\term 3\graph ml\Graph ML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(Helper.Version())

The version that you are using (0.9.20) is OLDER than the latest version (0.9.22) from PyPI. Please consider upgrading to the latest version.


In [3]:
renderer = "vscode"

## 2. Import OBJ Files

In [4]:
objects = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\base model.obj", selfMerge=True)
print("Objects:", objects)

Objects: [<topologic_core.Cluster object at 0x000002436D99E0F0>]


In [9]:
doors   = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\door.obj",   selfMerge=True)
windows = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\window.obj", selfMerge=True)

# Flatten clusters to individual faces and colour-code
door_faces   = []
window_faces = []

for ap in doors:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["purple", "door"]))
        door_faces.append(f)

for ap in windows:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["pink", "window"]))
        window_faces.append(f)

aperture_faces = door_faces + window_faces
print("Door faces:",    len(door_faces))
print("Window faces:",  len(window_faces))
print("Total apertures:", len(aperture_faces))

Door faces: 15
Window faces: 7
Total apertures: 22


## 3. Build CellComplex and Add Apertures

In [10]:
cells = Topology.Cells(objects[0])
print("Number of cells:", len(cells))

cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)
print("CellComplex:", cc)

Number of cells: 18
CellComplex: <topologic_core.CellComplex object at 0x000002437426D170>


In [11]:
cc = Topology.AddApertures(cc, aperture_faces, subTopologyType="face")
print("Apertures added:", len(aperture_faces))

Apertures added: 22


## 4. Show Geometry with Apertures

In [12]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster,
              faceColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 5. Build Fenestration Graph
Room cells → **red** nodes  
Door apertures → **brown** nodes  
Window apertures → **cyan** nodes  

For each CellComplex face that carries an aperture, an edge is drawn from the aperture node to every room that touches that face.

In [13]:
cc_cells = Topology.Cells(cc)

# --- Room vertices (one per cell, at internal point) ---
room_vertices = []
for i, c in enumerate(cc_cells):
    v = Topology.InternalVertex(c)
    if v:
        d = Dictionary.ByKeysValues(["type", "color", "size"], ["room", "red", 18])
        Topology.SetDictionary(v, d)
    room_vertices.append(v)

# --- Aperture vertices + edges ---
aperture_vertices = []
graph_edges       = []

cc_faces = Topology.Faces(cc)
for face in cc_faces:
    apers = Topology.Apertures(face)
    if not apers:
        continue

    # Rooms sharing this face
    adj_cells = Topology.SuperTopologies(face, cc, topologyType="cell")
    if not adj_cells:
        continue

    for ap in apers:
        # One node per aperture at its own centroid
        av = Topology.Centroid(ap)
        d_ap    = Topology.Dictionary(ap)
        color   = Dictionary.ValueAtKey(d_ap, "color") or "brown"
        ap_type = Dictionary.ValueAtKey(d_ap, "type")  or "door"
        d = Dictionary.ByKeysValues(["type", "color", "size"], [ap_type, color, 12])
        Topology.SetDictionary(av, d)
        aperture_vertices.append(av)

        # Connect this aperture node to each adjacent room node
        for adj_cell in adj_cells:
            adj_ctr = Topology.Centroid(adj_cell)
            best_v, best_d = None, float("inf")
            for rv in room_vertices:
                if rv is None:
                    continue
                dist = ((rv.X()-adj_ctr.X())**2 +
                        (rv.Y()-adj_ctr.Y())**2 +
                        (rv.Z()-adj_ctr.Z())**2) ** 0.5
                if dist < best_d:
                    best_d, best_v = dist, rv
            if best_v:
                e = Edge.ByVertices([av, best_v])
                if e:
                    graph_edges.append(e)

all_vertices = [v for v in room_vertices if v] + aperture_vertices
g_fen = Graph.ByVerticesEdges(all_vertices, graph_edges)
print("Room nodes:    ", len([v for v in room_vertices if v]))
print("Aperture nodes:", len(aperture_vertices))
print("Edges:         ", len(graph_edges))

Room nodes:     18
Aperture nodes: 22
Edges:          34


## 6. Assign Visual Attributes to Graph Edges

In [14]:
for e in Graph.Edges(g_fen):
    d = Dictionary.ByKeysValues(["width", "color"], [3, "black"])
    Topology.SetDictionary(e, d)

## 7. Show Fenestration Graph

In [15]:
Topology.Show(g_fen,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 8. Show Fenestration Graph Overlaid on Geometry

In [16]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster, g_fen,
              faceColorKey="color",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.15,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)